<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/01_schema_and_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# pytidb のスキーマとデータ型 (書籍ライブラリ)

このノートブックは **pytidb シリーズの第 1 回** です。 `TableModel` に TiDB 固有の型をどう表現するかを学びます。

## 学習目標
- `str` / `int` / `bool` / `datetime` などの基本型をマップする
- `TEXT`・`JSON`・`VECTOR` を明示的に指定する書き方を知る
- JSON 列にリストやネスト構造を入れて読み書きする

前提: `00_quickstart.ipynb` を実行済み。


## 1. 依存パッケージのインストール


In [ ]:
!pip install -q pytidb


## 2. TiDB Cloud Zero でクラスタを作成する


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-schema")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. 書籍テーブルを定義する

TiDB のデータ型は pytidb の `datatype` モジュールから `TEXT` / `JSON` / `VECTOR` が取れます。
Python の型ヒントと一緒に `Field(sa_type=...)` または `Field(sa_column=Column(...))` で指定します。


In [ ]:
from datetime import date
from pytidb import TiDBClient
from pytidb.schema import Field, TableModel, VectorField
from pytidb.datatype import JSON, TEXT

DATABASE = "books_demo"

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database=DATABASE,
    ensure_db=True,
)


class Book(TableModel):
    __tablename__ = "books"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    title: str = Field()                             # VARCHAR (<=255)
    summary: str = Field(sa_type=TEXT)               # 長文 → TEXT
    authors: list = Field(sa_type=JSON, default_factory=list)   # JSON 配列
    tags: dict = Field(sa_type=JSON, default_factory=dict)      # JSON オブジェクト
    published_on: date = Field()                     # DATE
    rating: float = Field(default=0.0)               # DOUBLE
    embedding: list[float] = VectorField(dimensions=8)   # VECTOR(8) (デモ用に 8 次元のダミー)


table = db.create_table(schema=Book, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 4. データを投入する

JSON 列には Python の `list` や `dict` をそのまま渡せます。
`VECTOR(8)` のダミーには手書きの 8 次元リストを入れます (本物の embedding は `03_vector_search.ipynb` で扱います)。


In [ ]:
samples = [
    Book(
        id=1,
        title="幻想機械の設計図",
        summary="19 世紀末のヨーロッパを舞台に、発明家と彼の幻想機械をめぐる冒険を描く長編小説。",
        authors=["白井 翔", "Mia Hoffmann"],
        tags={"genre": "fantasy", "language": "ja", "pages": 412},
        published_on=date(2021, 6, 15),
        rating=4.2,
        embedding=[0.1] * 8,
    ),
    Book(
        id=2,
        title="分散システム入門",
        summary="コンセンサスアルゴリズムとレプリケーションをゼロから解説する技術書。",
        authors=["中村 健二"],
        tags={"genre": "tech", "language": "ja", "pages": 268, "topics": ["raft", "paxos"]},
        published_on=date(2019, 11, 1),
        rating=4.6,
        embedding=[0.2] * 8,
    ),
    Book(
        id=3,
        title="Tea and Tectonics",
        summary="A memoir that weaves the history of Darjeeling tea with personal reflections on travel.",
        authors=["Priya Venkatesh"],
        tags={"genre": "memoir", "language": "en", "pages": 320},
        published_on=date(2023, 3, 28),
        rating=4.0,
        embedding=[0.3] * 8,
    ),
]
table.bulk_insert(samples)
print(f"投入完了。件数: {table.rows()}")


## 5. JSON 列を読み書きする

取得時は Python のオブジェクトとして戻ります。条件指定にはフィルタ辞書または SQL 的な式を使えます。


In [ ]:
books = table.query().to_pydantic()
for b in books:
    topics = b.tags.get("topics", [])
    print(f"#{b.id} {b.title}  著者={b.authors}  pages={b.tags.get('pages')}  topics={topics}")


## 6. JSON 列に対して条件を指定する

pytidb の `filter` は MongoDB 風の演算子 (`$eq`, `$gt`, `$lt`, `$in` など) を使えます。
JSON 列のキーへ降りるときは生 SQL を使うと分かりやすいです。


In [ ]:
# 単純なフィルタ: rating が 4.3 以上の本を取得
high_rated = table.query(filters={"rating": {"$gte": 4.3}}).to_pydantic()
print("評価 4.3 以上:")
for b in high_rated:
    print(f"  #{b.id} {b.title} ({b.rating})")

# JSON のネストキーで絞る場合は client.query (生 SQL) を使う
rows = db.query("""
    SELECT id, title, tags
    FROM books
    WHERE JSON_EXTRACT(tags, '$.genre') = 'tech'
""").to_rows()
print("\ngenre=tech の本:")
for r in rows:
    print(f"  {r}")


## 次のステップ

- `02_query_and_filter.ipynb` : クエリ・フィルタ・トランザクション
- `03_vector_search.ipynb` : VECTOR を実際の埋め込みで使う

## 追加実験

- `tags` に `"price": 1980` を加えて、価格帯で絞ってみる
- `published_on` で年ごとに絞り込み (`$gte` / `$lte`)
- `table.update(values={"rating": 4.8}, filters={"id": 2})` で更新
